<a href="https://colab.research.google.com/github/maitagora20-lab/smart-finance-assistant/blob/main/Maita_Finance_App.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import gradio as gr
import matplotlib.pyplot as plt
import re
import math

# ============================================================
# MAITA FINANCE SMART BUDDY
# Purpose: Analyse spending, give budget advice, and calculate savings goals
# ============================================================

def clean_amount(value):
    """Converts money values like 'Rs 1,200', '$300', or '-50' into numbers."""
    if pd.isna(value):
        return None

    value = str(value)
    value = re.sub(r"[^\d.\-"",]", "", value) # Removed the comma from the regex as it was causing issues

    try:
        return float(value)
    except ValueError:
        return None

def load_transactions(file):
    """Loads and prepares uploaded transaction CSV."""
    if file is None:
        return None, "Please upload your transaction CSV first."

    try:
        df = pd.read_csv(file.name)
    except Exception:
        return None, "I could not read this file. Please upload a valid CSV file."

    df.columns = df.columns.str.strip()

    required_columns = ["Category", "Amount"]
    missing = [col for col in required_columns if col not in df.columns]

    if missing:
        return None, f"Your CSV is missing: {missing}. It must include Category and Amount."

    df["Amount_Clean"] = df["Amount"].apply(clean_amount)
    df = df.dropna(subset=["Amount_Clean"])

    if df.empty:
        return None, "No valid amounts were found in your CSV."

    return df, "Your transactions were uploaded successfully."

def analyse_spending(df):
    """Creates spending summary and category analysis."""

    # Separate spending and refunds
    spending = df[df["Amount_Clean"] > 0]
    refunds = df[df["Amount_Clean"] < 0]

    # Calculate totals
    total_spending = spending["Amount_Clean"].sum()
    total_refunds = refunds["Amount_Clean"].sum()
    transaction_count = len(df)

    # Group spending by category
    category_table = (
        spending.groupby("Category")["Amount_Clean"]
        .agg(["sum", "mean", "count"])
        .reset_index()
        .rename(columns={
            "sum": "Total Spent",
            "mean": "Average Spend",
            "count": "Transactions"
        })
        .sort_values("Total Spent", ascending=False)
    )

    # Highest spending category
    if category_table.empty:
        top_category = "None"
        top_amount = 0
    else:
        top_category = category_table.iloc[0]["Category"]
        top_amount = category_table.iloc[0]["Total Spent"]

    # Dashboard summary
    summary = f"""
# Maita Finance Smart Buddy Dashboard

### Total Spending: Rs {total_spending:,.2f}
### Refunds / Money Back: Rs {abs(total_refunds):,.2f}
### Number of Transactions: {transaction_count}
### Biggest Spending Area: {top_category} — Rs {top_amount:,.2f}

---

## Smart Insight
Your biggest spending category is **{top_category}**.

This category is where most of your money is going.

## Recommendation
Try reducing spending in **{top_category}** by 10–15% next month.
"""

    return summary, category_table

def create_spending_chart(category_table):
    """Creates a spending chart."""

    if category_table is None or category_table.empty:
        return None

    fig, ax = plt.subplots(figsize=(8,5))

    ax.bar(category_table["Category"], category_table["Total Spent"])

    ax.set_title("Spending by Category")
    ax.set_xlabel("Category")
    ax.set_ylabel("Amount Spent")

    plt.xticks(rotation=35)
    plt.tight_layout()

    return fig



def money_chatbot(question, df):
    """Simple finance chatbot."""

    if not question or question.strip() == "":
        return "Ask me something about your spending or savings."

    q = question.lower()

    if df is not None:
        _, category_table = analyse_spending(df)

        if not category_table.empty:
            top_category = category_table.iloc[0]["Category"]
            top_amount = category_table.iloc[0]["Total Spent"]
            total_spending = category_table["Total Spent"].sum()
        else:
            top_category = "None"
            top_amount = 0
            total_spending = 0

    if "highest" in q or "biggest" in q:
        return f"""
Your highest spending category is **{top_category}**.

You spent **Rs {top_amount:,.2f}** there.
"""

    if "save" in q or "saving" in q:
        return """
Savings Advice:

- Reduce unnecessary spending
- Set a monthly savings target
- Track your expenses weekly
"""

    if "budget" in q:
        return """
Budget Advice:

- Separate needs and wants
- Review spending weekly
- Keep emergency savings
"""

    if "total" in q:
        return f"""
Your total spending is:

# Rs {total_spending:,.2f}
"""

    return """
I can help with:
- spending analysis
- savings advice
- budget advice
- highest spending category
"""

def savings_calculator(goal_amount, monthly_saving):
    """Calculates savings timeline."""

    try:
        goal_amount = float(goal_amount)
        monthly_saving = float(monthly_saving)

        if goal_amount <= 0:
            return "Goal amount must be greater than 0."

        if monthly_saving <= 0:
            return "Monthly saving amount must be greater than 0."

        months = math.ceil(goal_amount / monthly_saving)

        return f"""
To save Rs {goal_amount:,.2f}
while saving Rs {monthly_saving:,.2f} per month:

You need approximately {months} months.
"""

    except:
        return "Please enter valid numbers."